In [2]:
# ============================================================
# RESTAURANT - WEEK 1 SALES
# NLP + K-MEANS CLUSTERING
# JUPYTER NOTEBOOK
# ============================================================

# ============================================================
# CELL 1 - IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


# ============================================================
# CELL 2 - LOAD THE CSV FILE
# ============================================================

file_name = "Restaurant - Week 1 Sales.csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully.")
print("Shape of dataset:", df.shape)

display(df.head())


# ============================================================
# CELL 3 - BASIC DATASET INFORMATION
# ============================================================

print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# ============================================================
# CELL 4 - REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)


# ============================================================
# CELL 5 - CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("Cleaned column names:")
print(df.columns.tolist())


# ============================================================
# CELL 6 - DISPLAY DATA TYPES
# ============================================================

print("Data types:")
display(df.dtypes)


# ============================================================
# CELL 7 - IDENTIFY NUMERICAL AND TEXT COLUMNS
# ============================================================

numeric_columns = df.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

text_columns = df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numerical columns:")
print(numeric_columns)

print("\nText columns:")
print(text_columns)


# ============================================================
# CELL 8 - HANDLE MISSING VALUES
# ============================================================

# Fill numerical missing values with the median
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Fill text missing values with an empty string
for column in text_columns:
    df[column] = df[column].fillna("")

print("Missing values after cleaning:")
display(df.isnull().sum())


# ============================================================
# CELL 9 - BASIC STATISTICS
# ============================================================

if len(numeric_columns) > 0:
    print("Numerical summary:")
    display(df[numeric_columns].describe())


# ============================================================
# CELL 10 - CREATE A TEXT COLUMN FOR NLP
# ============================================================

# Combine all text columns into one column.
# This allows NLP to analyse the textual information in the dataset.

if len(text_columns) > 0:
    
    df["combined_text"] = df[text_columns].astype(str).agg(
        " ".join, axis=1
    )
    
else:
    
    # If there are no text columns, create an empty text column.
    df["combined_text"] = ""


# ============================================================
# CELL 11 - CLEAN TEXT FOR NLP
# ============================================================

import re

def clean_text(text):
    
    text = str(text).lower()
    
    # Remove numbers
    text = re.sub(r"\d+", " ", text)
    
    # Remove punctuation
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


df["clean_text"] = df["combined_text"].apply(clean_text)

print("Example of cleaned text:")
display(df[["combined_text", "clean_text"]].head())


# ============================================================
# CELL 12 - NLP: TF-IDF
# ============================================================

# TF-IDF converts text into numerical features.
#
# TF  = Term Frequency
# IDF = Inverse Document Frequency
#
# The result is a numerical matrix that can be used by K-Means.

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=500
)

X_text = tfidf.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", X_text.shape)


# ============================================================
# CELL 13 - DISPLAY IMPORTANT NLP WORDS
# ============================================================

feature_names = tfidf.get_feature_names_out()

print("Number of NLP features:", len(feature_names))

print("\nFirst 50 NLP features:")
print(feature_names[:50])


# ============================================================
# CELL 14 - PREPARE NUMERICAL SALES FEATURES
# ============================================================

# We exclude the text/NLP columns from the numerical data.

numeric_sales_columns = []

for column in numeric_columns:
    
    if column not in ["id"]:
        numeric_sales_columns.append(column)

print("Numerical features used:")
print(numeric_sales_columns)


# ============================================================
# CELL 15 - STANDARDISE NUMERICAL FEATURES
# ============================================================

if len(numeric_sales_columns) > 0:
    
    scaler = StandardScaler()
    
    X_numeric = scaler.fit_transform(
        df[numeric_sales_columns]
    )
    
    print("Numerical features standardised.")
    
else:
    
    X_numeric = np.empty((len(df), 0))
    
    print("No numerical features found.")


# ============================================================
# CELL 16 - COMBINE NLP + NUMERICAL FEATURES
# ============================================================

from scipy.sparse import hstack, csr_matrix

if X_numeric.shape[1] > 0:
    
    X_numeric_sparse = csr_matrix(X_numeric)
    
    X = hstack([
        X_text,
        X_numeric_sparse
    ])
    
else:
    
    X = X_text


print("Final feature matrix shape:", X.shape)


# ============================================================
# CELL 17 - ELBOW METHOD
# ============================================================

# The Elbow Method helps identify a suitable number
# of K-Means clusters.

inertia = []

max_k = min(10, len(df) - 1)

K_range = range(2, max_k + 1)

for k in K_range:
    
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    kmeans.fit(X)
    
    inertia.append(kmeans.inertia_)


plt.figure(figsize=(10, 6))

plt.plot(
    list(K_range),
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")

plt.grid(True)
plt.show()


# ============================================================
# CELL 18 - SILHOUETTE SCORE
# ============================================================

# Calculate silhouette scores for different K values.

silhouette_scores = []

for k in K_range:
    
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = kmeans.fit_predict(X)
    
    score = silhouette_score(X, labels)
    
    silhouette_scores.append(score)
    
    print(
        "K =", k,
        "| Silhouette Score =", round(score, 4)
    )


# ============================================================
# CELL 19 - PLOT SILHOUETTE SCORES
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    list(K_range),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for K-Means")

plt.grid(True)
plt.show()


# ============================================================
# CELL 20 - SELECT BEST NUMBER OF CLUSTERS
# ============================================================

best_k = list(K_range)[
    np.argmax(silhouette_scores)
]

print("Best number of clusters:", best_k)


# ============================================================
# CELL 21 - RUN FINAL K-MEANS MODEL
# ============================================================

kmeans_final = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans_final.fit_predict(X)

df["cluster"] = cluster_labels

print("K-Means clustering completed.")


# ============================================================
# CELL 22 - DISPLAY CLUSTERED DATA
# ============================================================

print("Clustered dataset:")

display(df.head(20))


# ============================================================
# CELL 23 - NUMBER OF RECORDS IN EACH CLUSTER
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("Number of records in each cluster:")

display(
    cluster_counts.to_frame(
        name="Number_of_Records"
    )
)


# ============================================================
# CELL 24 - CLUSTER DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

cluster_counts.plot(
    kind="bar"
)

plt.xlabel("Cluster")
plt.ylabel("Number of Records")
plt.title("Number of Restaurant Sales Records per Cluster")

plt.xticks(rotation=0)

plt.grid(axis="y")

plt.show()


# ============================================================
# CELL 25 - CLUSTER SUMMARY FOR NUMERICAL VARIABLES
# ============================================================

if len(numeric_sales_columns) > 0:
    
    cluster_summary = (
        df.groupby("cluster")[numeric_sales_columns]
        .mean()
        .round(2)
    )
    
    print("Average numerical values by cluster:")
    
    display(cluster_summary)


# ============================================================
# CELL 26 - MEDIAN VALUES BY CLUSTER
# ============================================================

if len(numeric_sales_columns) > 0:
    
    cluster_median = (
        df.groupby("cluster")[numeric_sales_columns]
        .median()
        .round(2)
    )
    
    print("Median numerical values by cluster:")
    
    display(cluster_median)


# ============================================================
# CELL 27 - FIND THE MOST IMPORTANT WORDS
# FOR EACH CLUSTER
# ============================================================

# Calculate the average TF-IDF score of words
# within each cluster.

if X_text.shape[1] > 0:
    
    X_text_dense = X_text.toarray()
    
    cluster_word_data = []
    
    for cluster in sorted(df["cluster"].unique()):
        
        cluster_indices = df["cluster"] == cluster
        
        mean_tfidf = X_text_dense[
            cluster_indices
        ].mean(axis=0)
        
        top_indices = mean_tfidf.argsort()[-10:][::-1]
        
        top_words = [
            feature_names[i]
            for i in top_indices
        ]
        
        cluster_word_data.append({
            "Cluster": cluster,
            "Top_Words": ", ".join(top_words)
        })
    
    cluster_words = pd.DataFrame(
        cluster_word_data
    )
    
    print("Most important words in each cluster:")
    
    display(cluster_words)


# ============================================================
# CELL 28 - NLP CLUSTER VISUALISATION USING SVD
# ============================================================

# Because TF-IDF can contain hundreds of dimensions,
# TruncatedSVD reduces the feature space to two dimensions
# for visualisation.

if X.shape[1] >= 2:
    
    svd = TruncatedSVD(
        n_components=2,
        random_state=42
    )
    
    X_2d = svd.fit_transform(X)
    
    plt.figure(figsize=(10, 7))
    
    scatter = plt.scatter(
        X_2d[:, 0],
        X_2d[:, 1],
        c=df["cluster"],
        s=60,
        alpha=0.7
    )
    
    plt.xlabel("SVD Component 1")
    plt.ylabel("SVD Component 2")
    plt.title("K-Means Clusters Using NLP + Sales Features")
    
    plt.colorbar(
        scatter,
        label="Cluster"
    )
    
    plt.grid(True)
    
    plt.show()


# ============================================================
# CELL 29 - VISUALISE NUMERICAL CLUSTERS
# ============================================================

# If there are at least two numerical variables,
# visualise the first two.

if len(numeric_sales_columns) >= 2:
    
    x_col = numeric_sales_columns[0]
    y_col = numeric_sales_columns[1]
    
    plt.figure(figsize=(10, 7))
    
    scatter = plt.scatter(
        df[x_col],
        df[y_col],
        c=df["cluster"],
        s=60,
        alpha=0.7
    )
    
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    
    plt.title(
        f"K-Means Clusters: {x_col} vs {y_col}"
    )
    
    plt.colorbar(
        scatter,
        label="Cluster"
    )
    
    plt.grid(True)
    
    plt.show()


# ============================================================
# CELL 30 - CLUSTER CENTRES
# ============================================================

print("K-Means model cluster centres:")

cluster_centres = kmeans_final.cluster_centers_

print(
    "Number of cluster centres:",
    len(cluster_centres)
)


# ============================================================
# CELL 31 - PREDICT CLUSTER FOR EACH RECORD
# ============================================================

df["predicted_cluster"] = kmeans_final.predict(X)

display(
    df[
        ["cluster", "predicted_cluster"]
    ].head(20)
)


# ============================================================
# CELL 32 - CHECK CLUSTER CONSISTENCY
# ============================================================

same_cluster = (
    df["cluster"] ==
    df["predicted_cluster"]
).all()

print(
    "Predicted clusters match original clusters:",
    same_cluster
)


# ============================================================
# CELL 33 - SORT DATA BY CLUSTER
# ============================================================

df_sorted = df.sort_values(
    by="cluster"
).reset_index(drop=True)

display(df_sorted.head(30))


# ============================================================
# CELL 34 - SAVE THE CLUSTERED DATASET
# ============================================================

output_file = "Restaurant_Week_1_Sales_KMeans_NLP.csv"

df.to_csv(
    output_file,
    index=False
)

print(
    "Clustered dataset saved as:",
    output_file
)


# ============================================================
# CELL 35 - SAVE CLUSTER SUMMARY
# ============================================================

if len(numeric_sales_columns) > 0:
    
    summary_file = "Restaurant_Week_1_Cluster_Summary.csv"
    
    cluster_summary.to_csv(
        summary_file
    )
    
    print(
        "Cluster summary saved as:",
        summary_file
    )


# ============================================================
# CELL 36 - FINAL RESULTS
# ============================================================

print("=" * 60)
print("FINAL K-MEANS + NLP RESULTS")
print("=" * 60)

print("\nNumber of records:", len(df))

print(
    "Number of features used:",
    X.shape[1]
)

print(
    "Optimal number of clusters:",
    best_k
)

print(
    "Best silhouette score:",
    round(
        max(silhouette_scores),
        4
    )
)

print("\nCluster sizes:")

display(
    df["cluster"]
    .value_counts()
    .sort_index()
    .to_frame("Records")
)

print("\nTop words by cluster:")

if "cluster_words" in locals():
    display(cluster_words)

print("\nAnalysis completed successfully.")


# ============================================================
# CELL 37 - OPTIONAL: SHOW EACH CLUSTER'S RECORDS
# ============================================================

for cluster_number in sorted(
    df["cluster"].unique()
):
    
    print("\n")
    print("=" * 60)
    print(f"CLUSTER {cluster_number}")
    print("=" * 60)
    
    cluster_data = df[
        df["cluster"] == cluster_number
    ]
    
    display(
        cluster_data.head(10)
    )

Libraries imported successfully.
Dataset loaded successfully.
Shape of dataset: (250, 2)


,Customer ID,Food ID
0,537,9
1,97,4
2,658,1
3,202,2
4,155,9


Column names:
['Customer ID', 'Food ID']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Customer ID  250 non-null    int64
 1   Food ID      250 non-null    int64
dtypes: int64(2)
memory usage: 4.0 KB

Missing values:


Customer ID    0
Food ID        0
dtype: int64


Duplicate rows: 2
Shape after removing duplicates: (248, 2)
Cleaned column names:
['customer_id', 'food_id']
Data types:


...

Numerical summary:


,customer_id,food_id
count,248.000000,248.000000
mean,455.725806,5.673387
std,289.817515,2.918976
min,3.000000,1.000000
25%,190.750000,3.000000
50%,415.500000,6.000000
75%,708.500000,8.000000
max,1000.000000,10.000000


Example of cleaned text:


,combined_text,clean_text
0,,
1,,
2,,
3,,
4,,


ValueError: empty vocabulary; perhaps the documents only contain stop words